Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital

**Aplicaciones Computacionales en Negocios**

# **Trabajo Práctico N°1 — Sistema de abordaje de pasajeros en un avión**

A lo largo de este trabajo práctico se nos pide modelar el sistema de abordaje de pasajero en un avión. A continuación, exponemos el proceso que llevamos a cabo para construir el modelo y el análisis sobre los resultados obtenidos a partir del mismo.

## **Ambiente físico**

A la hora de delimitar el espacio físico del modelo teníamos claro que correspondía al avión.

Luego, a partir de la consigna tuvimos claro que la idea de filas y columnas iba de la mano con una representación matricial del espacio físico. Bajo esta perspectiva, **un pasajero ocupa una casilla (o elemento) en la grilla (o matriz) del avión** a la que, por cierto, llamaremos $X \in \mathbb{R^{25\times5}}$. De aquí, se deduce con facilidad que $\text{col}_3(X)$ corresponde al pasillo central por el cual los pasajeros avanzan hacia sus asientos.

Finalmente, cabe aclarar que como bien nos indica la consigna, **nuestro reloj se mide en segundos**.

## **Agente**

Así como con el avión, aquí es claro que el agente corresponde a los pasajeros del avión. A continuación, listamos las acciones y estado de los agentes.

Desde el momento en que un pasajero se sube el avión, tiene un repertorio limitado de acciones:

- Avanzar.
- Levantarse.
- Guardar equipaje.
- Sentarse.

Asimismo, pueden adoptar algunos de los siguientes estados:
- Posición actual.
- Asiento.
- Asiento conseguido.
- Tiene carry-on (a mano).
- Sentado.
- Pasajero delante (en pasillo central).
- Pasajero en pasillo (ya estando sentado en fila junto al pasillo)

Vamos a asumir las siguientes reglas de comportamiento para los pasajeros:
- El pasajero que llega a su asiento vacío y no posee carry-on se demora de 4 a 12 segundos en sentarse.
- El pasajero que llega a su asiento vacío y que sí posee carry-on se demora de 12 a 32 segundos en sentarse.
- Cada pasajero que debe levantarse para habilitar la llegada al asiento agrega de 3 a 5 segundos al tiempo total de onboarding.
- Si se requiere guardar el carry-on y esperar para sentarse, necesariamente se debe guardar el carry-on primero para luego esperar a que se le deje pasar.



## **Modelo en acción**

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.figure as figure
from IPython.display import Image, display

import sys
from pathlib import Path

# Buscamos la carpeta del repositorio (la que tiene rutas.py). Arrancamos en la
# carpeta donde está abierto el notebook y vamos subiendo una carpeta a la vez.
# El "raiz != raiz.parent" es un freno de seguridad, si llegamos a la carpeta
# más alta de la computadora sin encontrarlo, cortamos en vez de seguir para
# siempre.
raiz = Path.cwd()
while not (raiz / 'rutas.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent

# Si terminamos de subir y aún así no apareció, avisamos con un mensaje claro
# en lugar de dejar que fallen los imports de más abajo sin explicación.
if not (raiz / 'rutas.py').exists():
    raise FileNotFoundError("No se encontró rutas.py subiendo desde " + str(Path.cwd()) + ". Abrí el notebook desde la carpeta del repositorio ACN-TP1.")

# Sumamos la carpeta del repo a la lista de lugares donde Python busca módulos.
sys.path.append(str(raiz))

# Al importarlo, rutas.py agrega por su cuenta las carpetas simulacion/ y
# analisis/ a esa lista. Por eso los imports que siguen encuentran los archivos,
# aunque acá no usemos rutas directamente.
import rutas

from ambiente_fisico import PlaneModel
from runner import run_simulations, save_results, run_and_show
from lectura_resultados import leer_carpeta, leer_resultados
from bonus_agente import BonusPassengerAgent as BonusAgent

In [3]:
METODOS = {'rand': 'Aleatorio', 'btf': 'Back-to-Front', 'wilma': 'WilMA', 'stfn': 'Steffen'}

METODO = 'rand'   # probar con 'btf', 'wilma' o 'stfn'
random.seed(42)   # para que la animación sea reproducible

model = PlaneModel(n=100, p=0.5, onboarding_method=METODO)


In [4]:
N_SIMULACIONES = 100
N_PASAJEROS = 100
P_CARRYON = 0.65

In [ ]:
results = run_simulations(n_simulations=N_SIMULACIONES, n_passengers=N_PASAJEROS, p_carryon=P_CARRYON)
save_results(results = results, n_passengers=N_PASAJEROS, p_carryon=P_CARRYON)

In [ ]:
# Prueba de visualización
gif = run_and_show(N_PASAJEROS, 0.5, method='rand')
display(Image(gif))

## **Análisis estadístico de los resultados**

In [ ]:
# Probamos lectura de carpetas y lectura de archivos
r = leer_resultados('../resultados/grandes_simulaciones/p_08.txt')
n_bins = 200
data_btf = r.tiempos('btf')
data_rand = r.tiempos('rand')
data_wilma = r.tiempos('wilma')
data_stfn = r.tiempos('stfn')
colores = [(140/255, 114/255, 132/255, 0.5),(0, 78/255, 100/255, 0.5),(94/255, 11/255, 21/255, 0.5),(230/255, 170/255, 104/255, 0.5)]

plt.title("Histogramas de las diferentes políticas para p = 0.8 con 200 bins")
plt.xlabel("Tiempo total de onboarding en segundos")
plt.ylabel("Muestras que caen en el intervalo")

plt.show()

In [ ]:
b = leer_carpeta()

## **Bonus: exploración libre**

Ahora probamos el bonus.

In [5]:
model_bonus = PlaneModel(n=100, p=0.5, onboarding_method=METODO, agent_class=BonusAgent) # type: ignore

In [ ]:
results_bonus = run_simulations(n_simulations=N_SIMULACIONES, n_passengers=N_PASAJEROS, p_carryon=P_CARRYON, agent_class=BonusAgent) # type: ignore
save_results(results=results_bonus, n_passengers=N_PASAJEROS, p_carryon=P_CARRYON, etiqueta='bonus')

In [ ]:
gif_bonus = run_and_show(N_PASAJEROS, 0.5, method='rand', agent_class=BonusAgent) # type: ignore
display(Image(gif_bonus))